# 04b — Live data-quality review with Gemini

Runs the Data Quality Reviewer agent against the real Gemini API, step by step.
**Run the cells in order** (or *Run All*). Every step checks that the earlier ones ran and tells you which is missing.

**Before you start (once):**
1. Terminal, in the project folder: `cp .env.example .env`
2. Open `.env` and set `GEMINI_API_KEY=your-key` (no quotes). Get a key in Google AI Studio → *Get API key*.
3. Data generated: `python scripts/generate_data.py`
4. Select the `.venv` kernel. **After editing `.env`, restart the kernel and Run All.**

Cost: Step 3 is a tiny call; Step 4 is 1 call (2 if a revision is needed). Without a key every live step is skipped cleanly.

## Step 1 — Setup (no API calls)
Expected: `Validation : {'pass': 30, 'fail': 20, ...} -> fail`, `Baseline : not_fit_for_use | 7 findings`, `STEP 1 OK`.

In [ ]:
# STEP 1 - Setup: project root, imports, data, validation report, baseline review (no API calls)
import os, sys, json
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("pyproject.toml not found - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

import pandas as pd
from typing import Literal
from pydantic import BaseModel
from analytics_agent.config import load_settings
from analytics_agent.data_generation.generator import GenerationConfig
from analytics_agent.data_quality import validate_tables, load_tables_from_files
from analytics_agent.agents.data_quality_reviewer import (
    DataQualityReviewer, deterministic_review, check_review, review_to_markdown)
from analytics_agent.llm import GeminiStructuredClient, LLMError
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 100); pd.set_option("display.max_columns", 20)

def require(*names):
    """Stop with a clear message if an earlier step has not been run in this kernel session."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"Missing {missing}. Run the earlier steps first, in order (or use Run All).")

settings = load_settings(require_project=False)
AS_OF = GenerationConfig().as_of_ts
corrupted = load_tables_from_files("corrupted", "parquet")
rep_bad = validate_tables(corrupted, AS_OF, dataset_label="corrupted")
base = deterministic_review(rep_bad)
catalog = json.loads(Path("data/manifests/defect_catalog.json").read_text())

# Results of later steps start empty, so later cells never hit NameError.
RUN_LIVE, llm, live = False, None, None

print("Project root :", root)
print("Validation   :", rep_bad.counts(), "->", rep_bad.overall_status)
print("Baseline     :", base.overall_assessment, "|", len(base.findings), "findings")
print("STEP 1 OK")

## Step 2 — Check the configuration
Expected: `.env file : found`, `GEMINI_API_KEY : set (NN characters)`, `Model : gemini-3.8-flash`.

In [ ]:
# STEP 2 - Check the Gemini configuration (the key value is never printed)
require("settings")
import importlib.metadata as md
key = os.environ.get("GEMINI_API_KEY", "")
print(".env file      :", "found" if (root / ".env").exists() else "MISSING -> run in terminal: cp .env.example .env")
print("GEMINI_API_KEY :", f"set ({len(key)} characters)" if settings.gemini_key_present else "NOT SET")
print("Model          :", settings.gemini.model)
print("google-genai   :", md.version("google-genai"))

RUN_LIVE = settings.gemini_key_present
print("STEP 2 OK -", "live steps will run" if RUN_LIVE else
      "no key: live steps will be skipped (add GEMINI_API_KEY to .env, restart kernel, Run All)")

## Step 3 — Smoke test
Expected: `Reply : {'status': 'ok', ...}` and `STEP 3 OK`. If it fails, the cell prints the exact fix. **Do not continue until this passes.**

In [ ]:
# STEP 3 - Smoke test: one tiny call checks key + model + structured output
require("settings", "RUN_LIVE")

class Ping(BaseModel):
    status: Literal["ok"]
    note: str

FIXES = {
    "not_configured": "Set GEMINI_API_KEY in .env and restart the kernel.",
    "auth": "Key invalid or restricted - create a new key in Google AI Studio.",
    "model_not_found": "Set GEMINI_MODEL in .env to a model from ai.google.dev/gemini-api/docs/models.",
    "quota": "Quota exhausted - check usage/billing in Google AI Studio.",
    "rate_limit": "Too many requests - wait a minute and re-run this step.",
    "network": "Cannot reach generativelanguage.googleapis.com - check internet/proxy/firewall/VPN.",
    "timeout": "Timed out - re-run, or raise gemini.timeout_seconds in configs/config.yaml.",
    "invalid_output": "Reply did not match the schema - re-run once; if repeated, try another model.",
}

llm = None
if RUN_LIVE:
    try:
        llm = GeminiStructuredClient(settings.gemini.model, timeout_s=settings.gemini.timeout_seconds,
                                     max_retries=settings.gemini.max_retries)
        ping, meta = llm.generate(system_instruction="Reply with the requested JSON only.",
                                  prompt='Return status "ok" and a five-word note.', schema=Ping,
                                  max_output_tokens=512)
        print("Reply   :", ping.model_dump())
        print("Latency :", meta.latency_s, "s | attempts", meta.attempts,
              "| tokens in/out", meta.input_tokens, "/", meta.output_tokens)
        print("STEP 3 OK - Gemini works")
    except LLMError as e:
        RUN_LIVE, llm = False, None
        print(f"STEP 3 FAILED: {e.kind} (HTTP {e.code})")
        print("Message:", e.message[:300])
        print("Fix    :", FIXES.get(e.kind, "See the message above."))
else:
    print("STEP 3 skipped (no API key).")

## Step 4 — Live review
Outcome meanings: **accepted** = first answer passed all guardrails; **revised** = first answer rejected (reasons printed), corrected answer passed; **escalated** = failed twice or API error → deterministic review + human review. Expected overall: `not_fit_for_use`.

In [ ]:
# STEP 4 - Live review of the corrupted dataset (1 call, at most 1 revision)
require("rep_bad", "RUN_LIVE", "llm")
live = None
if RUN_LIVE and llm is not None:
    before = json.dumps(rep_bad.to_dict()["results"])
    live = DataQualityReviewer(llm, max_revisions=1, temperature=settings.gemini.temperature,
                               max_output_tokens=settings.gemini.max_output_tokens).review(rep_bad)
    assert json.dumps(rep_bad.to_dict()["results"]) == before, "the reviewer must never change rule statuses"

    print("Outcome      :", live.outcome, "   (accepted / revised / escalated)")
    print("Reviewer     :", live.reviewer)
    print("Guardrails   :", "passed" if live.guardrail.passed else "FAILED")
    print("Attempts     :", live.attempts, "| human review:", live.requires_human_review)
    if live.error:
        print("Note         :", live.error)
    for i, reasons in enumerate(live.rejected_attempts, 1):
        print(f"Rejected attempt {i}:")
        for r in reasons:
            print("   -", r)
    for m in live.llm_meta:
        if m:
            print(f"Call         : {m.latency_s}s, tokens in/out/thinking {m.input_tokens}/{m.output_tokens}/{m.thought_tokens}, errors {m.errors}")
    print("\nOverall      :", live.review.overall_assessment)
    print(live.review.summary)
    print("STEP 4 OK")
else:
    print("STEP 4 skipped - Step 3 did not pass (see its output).")

## Step 5 — Inspect the review
Shows the live review if Step 4 ran, otherwise the baseline. Expected comparison: both `not_fit_for_use`, defects covered `15/15`, guardrails passed `True`. Gemini groups findings its own way, so titles differ from the baseline.

In [ ]:
# STEP 5 - Inspect the review: findings table, one finding in full, comparison with baseline
require("rep_bad", "base", "catalog", "live")

def coverage(review):
    cited = {rid for f in review.findings for rid in f.rule_ids}
    return f"{sum(d['detection_rule'] in cited for d in catalog)}/{len(catalog)}"

shown = live.review if live is not None else base
label = f"LIVE review ({live.reviewer}, {live.outcome})" if live is not None else "BASELINE review (no live review available)"
print(label, "\n")

display(pd.DataFrame([{"title": f.title, "severity": f.severity, "classification": f.classification,
                       "rules": ", ".join(f.rule_ids), "human review": f.requires_human_review}
                      for f in shown.findings]))

N = 0   # change to see another finding
f = shown.findings[N]
print(f"\nFinding {N}: [{f.severity}] {f.title} ({f.classification})")
for e in f.evidence:
    print("   evidence:", e)
print("   impact  :", f.business_impact)
print("   metrics :", ", ".join(f.affected_metrics))
print("   action  :", f.recommended_action, "\n")

rows = [{"review": "baseline (no LLM)", "overall": base.overall_assessment, "findings": len(base.findings),
         "defects covered": coverage(base), "guardrails passed": check_review(base, rep_bad).passed}]
if live is not None:
    rows.append({"review": f"live ({live.reviewer})", "overall": live.review.overall_assessment,
                 "findings": len(live.review.findings), "defects covered": coverage(live.review),
                 "guardrails passed": live.guardrail.passed})
display(pd.DataFrame(rows))
print("STEP 5 OK")

## Step 6 — Save
Writes `reports/dq/<run_id>_review.md` and `.json`. Open the `.md` in VS Code and press Ctrl+Shift+V.

In [ ]:
# STEP 6 - Save the review to reports/dq/ (Markdown + JSON)
require("rep_bad", "live")
final = live if live is not None else DataQualityReviewer(llm=None).review(rep_bad)
out_dir = root / "reports" / "dq"
out_dir.mkdir(parents=True, exist_ok=True)
stem = out_dir / f"{rep_bad.run_id}_review"
stem.with_suffix(".md").write_text(review_to_markdown(final))
stem.with_suffix(".json").write_text(json.dumps({
    "run_id": rep_bad.run_id, "outcome": final.outcome, "reviewer": final.reviewer,
    "guardrails_passed": final.guardrail.passed, "requires_human_review": final.requires_human_review,
    "rejected_attempts": final.rejected_attempts, "llm_calls": [vars(m) for m in final.llm_meta if m],
    "review": final.review.model_dump()}, indent=2))
print("Saved:", stem.with_suffix(".md").relative_to(root))
print("Saved:", stem.with_suffix(".json").relative_to(root))
print("STEP 6 OK - outcome:", final.outcome, "| reviewer:", final.reviewer)

## Step 7 — Optional extra checks
Set a flag to `True` and re-run this cell.

In [ ]:
# STEP 7 (optional) - Extra live checks. Turn on with the flags below.
require("settings", "llm")
RUN_ERROR_TESTS = False    # wrong key + unknown model: fail instantly, not billed
RUN_STALE_REVIEW = False   # one more API call: a warning-only report -> expect 'use_with_caveats'

if RUN_ERROR_TESTS:
    for label, model, kw in [("invalid key", settings.gemini.model, {"api_key": "invalid-key-for-testing"}),
                             ("unknown model", "gemini-does-not-exist", {})]:
        try:
            GeminiStructuredClient(model, max_retries=0, **kw).generate(
                system_instruction="Reply with JSON.", prompt="ping", schema=Ping, max_output_tokens=64)
            print(f"{label:<14} -> unexpectedly succeeded")
        except LLMError as e:
            print(f"{label:<14} -> {e.kind} (HTTP {e.code})   expected: auth / model_not_found")

if RUN_STALE_REVIEW and llm is not None:
    clean = load_tables_from_files("clean", "parquet")
    rep_stale = validate_tables(clean, AS_OF + pd.Timedelta(days=30), dataset_label="clean (stale snapshot)")
    res = DataQualityReviewer(llm, max_revisions=1).review(rep_stale)
    print(rep_stale.counts(), "->", res.outcome, "|", res.review.overall_assessment)
    for f in res.review.findings:
        print(f"   [{f.severity}] {f.classification}: {f.title} -> {', '.join(f.rule_ids)}")

if not (RUN_ERROR_TESTS or RUN_STALE_REVIEW):
    print("STEP 7 skipped (both flags are False).")